# Agenda:

1. Review of working with files
2. Files and data structures -- how are they connected? Why do we care?
3. The `with` construct
4. Writing to files as well

# File basics

A file is just a collection of bits (1s and 0s). However, if we go up a little bit in terms of abstraction, we can think of a file as a collection of data structures.

A program can store data structures in a file to (a) share it (b) backup or (c) have it around after rebooting.

You *can* store binary versions, bit-for-bit, of your data structures. However, almost no one does this. Rather, we typically translate our data structures into something that's more file-friendly. When we read the data back into a program, we turn it from this file-friendly version into a data structure friendly version.

There are a few different techniques for making this translation:

- Just use a text file. (That's basically what we're going to do here.) Do some simple approximation of your data structure into a string, and store that string in a file. When you read the string back from the file, you'll then turn it back into your data structure. Advantage: It's easy, when the data is simple. Disadvantage: There's lots of room for ambiguity.

- You can use an existing file format that someone else (or some organization) has created. A very popular file format nowadays is JSON (JavaScript Object Notation). This is a standard way to go from text to data structures, and from data structures to text in many different programming languages. (It's not only for JavaScript.) For Python people, this is super easy, because you can think of JSON as a Python dict. The values in the dicts are then our lists, strings, and integers.

- Before JSON, people used a format called XML. Today, XML is still used, but it's seen as old and complex, especially when compared with JSON. But you can definitely find many ways to convert your Python values into XML, and then again from XML back into Python.

- Even older (and more common, even today) is CSV, comma-separated values. Each line in the file contains one record, and commas (typically) separate the fields in each record. So you'll have a file that looks like

    first_name,last_name,shoe_size
    Reuven,Lerner,46
    John,Doe,42

CSV is still very common and very popular, even though it has a lot of the same ambiguity problems as regular text files. 


# How can I read from a file?

The easiest, fastest, and most common way to read from a file in Python is by iterating over it in a `for` loop:

- The loop stops automatically when you get to the end of the file
- We get, with each iteration, one line of the file
- Each line comes as a string, ending with `'\n'`
- In each iteration, you do what you want with the file

In [1]:
# let's read the passwd.txt file (from Unix) using a loop

f = open('passwd.txt')   # open the file -- ask the OS for a connection to it

for one_line in f:       # iterate over the file, one line at a time
    print(one_line)      # print the current line

nobody:*:-2:-2:Unprivileged User:/var/empty:/usr/bin/false

root:*:0:0:System Administrator:/var/root:/bin/sh

daemon:*:1:1:System Services:/var/root:/usr/bin/false

_uucp:*:4:4:Unix to Unix Copy Protocol:/var/spool/uucp:/usr/sbin/uucico

_taskgated:*:13:13:Task Gate Daemon:/var/empty:/usr/bin/false

_networkd:*:24:24:Network Services:/var/networkd:/usr/bin/false

_installassistant:*:25:25:Install Assistant:/var/empty:/usr/bin/false

_lp:*:26:26:Printing Services:/var/spool/cups:/usr/bin/false

_postfix:*:27:27:Postfix Mail Server:/var/spool/postfix:/usr/bin/false

_scsd:*:31:31:Service Configuration Service:/var/empty:/usr/bin/false

_ces:*:32:32:Certificate Enrollment Service:/var/empty:/usr/bin/false

_appstore:*:33:33:Mac App Store Service:/var/db/appstore:/usr/bin/false

_mcxalr:*:54:54:MCX AppLaunch:/var/empty:/usr/bin/false

_appleevents:*:55:55:AppleEvents Daemon:/var/empty:/usr/bin/false

_geod:*:56:56:Geo Services Daemon:/var/db/geod:/usr/bin/false

_devdocs:*:59:59:Developer

Each line in the passwd file contains a record for one user, with the following fields:

- username
- `*` where the password used to be
- user ID number
- group ID number
- user's real-world name
- user's home directory
- user's "shell," or the command interpreter used when the person logs in

In [3]:
# let's print all of the usernames on this system

f = open('passwd.txt')   # open the file -- ask the OS for a connection to it

for one_line in f:                 # iterate over the file, one line at a time
    fields = one_line.split(':')   # str.split returns a list of strings, based on the string, using our argument as a separator
    print(fields[0])               # grab the first item from the list, and print it -- that'll be the username

nobody
root
daemon
_uucp
_taskgated
_networkd
_installassistant
_lp
_postfix
_scsd
_ces
_appstore
_mcxalr
_appleevents
_geod
_devdocs
_sandbox
_mdnsresponder
_ard
_www
_eppc
_cvs
_svn
_mysql
_sshd
_qtss
_cyrus
_mailman
_appserver
_clamav
_amavisd
_jabber
_appowner
_windowserver
_spotlight
_tokend
_securityagent
_calendar
_teamsserver
_update_sharing
_installer
_atsserver
_ftp
_unknown
_softwareupdate
_coreaudiod
_screensaver
_locationd
_trustevaluationagent
_timezone
_lda
_cvmsroot
_usbmuxd
_dovecot
_dpaudio
_postgres
_krbtgt
_kadmin_admin
_kadmin_changepw
_devicemgr
_webauthserver
_netbios
_warmd
_dovenull
_netstatistics
_avbdeviced
_krb_krbtgt
_krb_kadmin
_krb_changepw
_krb_kerberos
_krb_anonymous
_assetcache
_coremediaiod
_launchservicesd
_iconservices
_distnote
_nsurlsessiond
_displaypolicyd
_astris
_krbfast
_gamecontrollerd
_mbsetupuser
_ondemand
_xserverdocs
_wwwproxy
_mobileasset
_findmydevice
_datadetectors
_captiveagent
_ctkd
_applepay
_hidd
_cmiodalassistants
_analyticsd
_fps

In [4]:
# another file: mini-access-log.txt

for one_line in open('mini-access-log.txt'):
    print(one_line.strip())

67.218.116.165 - - [30/Jan/2010:00:03:18 +0200] "GET /robots.txt HTTP/1.0" 200 99 "-" "Mozilla/5.0 (Twiceler-0.9 http://www.cuil.com/twiceler/robot.html)"
66.249.71.65 - - [30/Jan/2010:00:12:06 +0200] "GET /browse/one_node/1557 HTTP/1.1" 200 39208 "-" "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)"
65.55.106.183 - - [30/Jan/2010:01:29:23 +0200] "GET /robots.txt HTTP/1.1" 200 99 "-" "msnbot/2.0b (+http://search.msn.com/msnbot.htm)"
65.55.106.183 - - [30/Jan/2010:01:30:06 +0200] "GET /browse/one_model/2162 HTTP/1.1" 200 2181 "-" "msnbot/2.0b (+http://search.msn.com/msnbot.htm)"
66.249.71.65 - - [30/Jan/2010:02:07:14 +0200] "GET /browse/browse_applet_tab/2593 HTTP/1.1" 200 10305 "-" "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)"
66.249.71.65 - - [30/Jan/2010:02:10:39 +0200] "GET /browse/browse_files_tab/2499?tab=true HTTP/1.1" 200 446 "-" "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)"
66.249.65.12 - - [30/J

# Exercise: IP address count

1. Define `counts`, an empty dictionary.
2. Go through `mini-access-log.txt`, one line at a time.
3. Grab the IP address from the start of the line.
4. Record this IP address:
    - If this is the first time you're seeing the address, add the address to `counts` as a key, and 1 should be the value.
    - If you have seen this IP address before, then just add 1 to the existing count for this IP address in the dict.
5. Iterate over the dict when you're done, printing each key (IP address) and value (number of times it appears in the file).

https://practice.lernerpython.com/classroom/ce06963a8d/ex-25

In [13]:
counts = {}    # this is a dict, thanks to the use of {} and not [] (a list) or () (a tuple)

f = open('mini-access-log.txt')

for one_line in f:
    ip_address = one_line.split()[0]  # splitting on nothing means: all whitespace, of any sort, any combination, and any length

    if ip_address in counts:         # have we seen this IP address before?
        counts[ip_address] += 1      # ... if so, then I can add 1 to it
    else:
        counts[ip_address] = 1       # if not, then add a new key-value pair to the dict with the IP address and a starter value of 1

for key, value in counts.items():    # dict.items() returns (key, value) for each pair -- which we then retrieve with unpacking in the loop
    print(f'{key}:\t{value}')

67.218.116.165:	2
66.249.71.65:	3
65.55.106.183:	2
66.249.65.12:	32
65.55.106.131:	2
65.55.106.186:	2
74.52.245.146:	2
66.249.65.43:	3
65.55.207.25:	2
65.55.207.94:	2
65.55.207.71:	1
98.242.170.241:	1
66.249.65.38:	100
65.55.207.126:	2
82.34.9.20:	2
65.55.106.155:	2
65.55.207.77:	2
208.80.193.28:	1
89.248.172.58:	22
67.195.112.35:	16
65.55.207.50:	3
65.55.215.75:	2


Technical debt -- problems with software that will require an investment to undo. These are often deep-seated issues with the design of the software that is often quite old and fragile.

In [14]:
# .doc -> .docx

In [15]:
# we know that we cannot add an integer and a string, much as we might want to

5 + 'a'

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [16]:
# can I, however, multiply an int by a string?
# yes!

5 * 'a'

'aaaaa'

In [17]:
10 * 'abc'

'abcabcabcabcabcabcabcabcabcabc'

In [19]:
counts = {}    # this is a dict, thanks to the use of {} and not [] (a list) or () (a tuple)

f = open('mini-access-log.txt')

for one_line in f:
    ip_address = one_line.split()[0]  # splitting on nothing means: all whitespace, of any sort, any combination, and any length

    if ip_address in counts:         # have we seen this IP address before?
        counts[ip_address] += 1      # ... if so, then I can add 1 to it
    else:
        counts[ip_address] = 1       # if not, then add a new key-value pair to the dict with the IP address and a starter value of 1

for key, value in counts.items():    # dict.items() returns (key, value) for each pair -- which we then retrieve with unpacking in the loop
    print(f'{key}:\t{value * "x"}')

67.218.116.165:	xx
66.249.71.65:	xxx
65.55.106.183:	xx
66.249.65.12:	xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
65.55.106.131:	xx
65.55.106.186:	xx
74.52.245.146:	xx
66.249.65.43:	xxx
65.55.207.25:	xx
65.55.207.94:	xx
65.55.207.71:	x
98.242.170.241:	x
66.249.65.38:	xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
65.55.207.126:	xx
82.34.9.20:	xx
65.55.106.155:	xx
65.55.207.77:	xx
208.80.193.28:	x
89.248.172.58:	xxxxxxxxxxxxxxxxxxxxxx
67.195.112.35:	xxxxxxxxxxxxxxxx
65.55.207.50:	xxx
65.55.215.75:	xx


# Exercise: Word count

Unix systems come with lots of little utilities, and one of them is known as `wc`, short for "word count." The idea is that if you run `wc` against a text file, it'll give you:

- Number of lines
- Number of words (separated by spaces)
- Number of characters (including spaces, newlines, punctuation, etc.)

Write your own version of `wc`, such that you will ask the user to enter the name of a file. You can then go through that file, collecting data about it -- number of lines, number of words, and number of characters.

When you're done iterating through the file, print your results/counts for each of these measures.

There is a file, called, `wcfile.txt`, that you can use.

https://practice.lernerpython.com/classroom/ce06963a8d/ex-28

In [20]:
# Jupyter trick -- ! at the start of the line lets you run something in Unix (if you're using it)

!cat wcfile.txt

This is a test file.

It contains 28 words and 20 different words.

It also contains 165 characters.

It also contains 11 lines.

It is also self-referential.

Wow!


In [27]:
# setup our dict with count info
counts = {'lines': 0,
          'words': 0,
          'chars': 0}

# go through the file, one line at a time, and count lines/words/chars
for one_line in open('wcfile.txt'):
    counts['lines'] += 1  
    counts['chars'] += len(one_line)   # count the characters in this line, then add to the existing character count
    counts['words'] += len(one_line.split())  # count the word on this line, then add that to the existing word count

# report back to the user
for key, value in counts.items():
    print(f'{key}: {value}')

lines: 11
words: 28
chars: 165


# Whatever happened to closing the file?

I admit: I tend not to close files very often, at least when I'm reading from them:

- If I'm only dealing with a small number of files (including only one), then it doesn't really hurt anyone
- If I'm only reading from the file, it hurts less -- writing to a file really does require closing
- It used to be that an open file consumed resources, and that wasted a computer's abilities

It is still a good idea to close files when you're done with them. It's cleaner, nicer, and when we get to writing to files, it's especially important.

How can we close a file? Answer: Invoke the `close` method on a file object.

In [28]:
f = open('wcfile.txt')
f.readline()   # returns the next line in a file as a string

'This is a test file.\n'

In [29]:
f.close()  # once I have closed a file, I can't do anything more with it!

In [30]:
f.readline()  # read another line!

ValueError: I/O operation on closed file.

In [31]:
f.close()  # what if I try to close it again?

In [32]:
f   # show me the object's printed representation

<_io.TextIOWrapper name='wcfile.txt' mode='r' encoding='UTF-8'>

# The standard paradigm

1. `open` the file
2. Do something with the file
3. `close` the file

# If this is so common, why do we have to work so hard?

Why are we closing the file? Why can't Python do it for us automatically when we're done working with the file?

This is where the `with` construct in Python comes into play. `with`, for files, lets us open a file, then do things with it, and then it automatically closes the file when you're done.

In [33]:
# rewrite our earlier code using "with"

with open('wcfile.txt') as f:   # open the file and assign to f, and prepare for an indented block
    print(f.readline())         # do whatever you want with the file

# when the "with" block ends, the file is automatically closed    

This is a test file.



In [34]:
f.closed

True

In [35]:
f.readline()

ValueError: I/O operation on closed file.

# Writing to files

We might, sometimes, want to write to files! A few things to note about that:

- Generally speaking, you can either read from or write to a file at a given time. Even if you can, I strongly suggest you don't.
- So far, we have not told `open` that we want to read from the file. That is its assumption and default. However, we could have said `open(filename, 'r')` to get the same functionality, reading from the file.
- If we want to write to a file, we say `open(filename, 'w')`. This tells Python to open the file for writing, and only for writing. You cannot read from a file while you're writing to it.
- You write to a file by invoking the `write` method on the file object. Pass it a string, and realize that it does *not* automatically add `'\n'` to the end of each write.
- Because of buffering, you really want to close the file after writing -- or just use `with` and save yourself frustration.

In [37]:
with open('myfile.txt', 'w') as f:    # open the file for writing, and assign it to f
    f.write('hello!\n')               # write two sentences to the file
    f.write('hello again!\n')

# thanks to "with", at the end of this block, the file is closed

In [38]:
!cat myfile.txt

hello!
hello again!


# Two important things about files and `with`

1. If you don't close the file, then it might take a long time for your data to actually appear there. That's because disks are so much slower than your software that an operating system won't really write data to disk when you ask it to. Rather, it'll bunch it into larger "write" commands, and write them all at once. Until then, the data is stored in a "memory buffer," which as it fills up gets ready to write. Without `close` or the magic close from `with`, you don't know when the buffer will fill, and thus the data will be written to disk.

2. If you open a file for writing with `'w'`, any content in the file is destroyed. You can also open with `'a'`, which appends the end of the file. Or you can use `'x'`, which gives an error if the file already exists.

# Exercise: Dict to config

1. Define a small dict.
2. Using `with`, `open`, and the `write` method, write the dict to a file, such that each pair is on a separate line, and the keys are separated from the values with `=`.
3. In the end, the file should contain all key-value pairs that were in your dict.
4. Read the file and check that it matches.

https://practice.lernerpython.com/classroom/ce06963a8d/ex-29